In [1]:
# %% 셀 1: 모델 + 데이터 로드 (Model 1 — merged layout)
import os, json, random, hashlib, colorsys, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
FRAME_DIR = "./data/2_frame_files"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_TEXT_LEN = 200
SCALAR_DIM = 9
TIME_DIM = 3
D_MODEL = 256
D_PIX = 128
N_HEADS = 8
N_LAYERS_INST = 4
D_FF = 512
DROPOUT = 0.0
DEBUG_CHANNELS = 67
CKPT_PATH = "./model/8_layout_1_merged/best.pt"

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels_all = sorted(channel_set)

# 채널 서브샘플링 재현 (train과 동일)
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels_all, min(DEBUG_CHANNELS, len(channels_all)))
sampled_set = set(sampled_channels)
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

eval_samples = []
for ch in sorted(by_channel.keys()):
    if ch not in sampled_set:
        continue
    ch_samples = by_channel[ch]
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])

print(f"✅ 채널: {len(channels)}개 (sampled from {len(channels_all)})")
print(f"✅ eval 영상: {len(eval_samples):,}개")


# ── 모델 정의 (train과 동일) ──
class MergedLayoutModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, d_pix=D_PIX,
                 n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.d_pix = d_pix

        self.ch_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(nn.Linear(5, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))

        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.channel_emb = nn.Embedding(n_channels, d_model)

        self.time_proj = nn.Linear(TIME_DIM, d_model)
        self.count_proj = nn.Sequential(
            nn.Linear(1, d_model // 4), nn.GELU(),
            nn.Linear(d_model // 4, d_model))

        self.frame_proj = nn.Sequential(
            nn.Linear(d_model, d_pix), nn.GELU(),
            nn.Linear(d_pix, d_pix))
        self.spatial_codebook = nn.Parameter(torch.randn(d_pix, GRID_H, GRID_W) * 0.02)
        self.heatmap_bias = nn.Parameter(torch.zeros(GRID_H, GRID_W))
        self.logit_scale = nn.Parameter(torch.tensor(1.0 / math.sqrt(d_pix)))

    def encode_inst(self, channel_ids, inst_diff_ch, inst_diff_vid,
                    inst_diff_stt, inst_scalars):
        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([
            inst_scalars[..., 0:1],
            inst_scalars[..., 5:8],
            inst_scalars[..., 8:9]], dim=-1)

        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)

        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1))
        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        inst_tokens = inst_tokens + ch_emb

        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask
        inst_tokens = self.inst_self_attn(inst_tokens, src_key_padding_mask=inst_pad)
        return inst_tokens, inst_mask

    def aggregate_frames(self, inst_tokens, active_per_frame, time_feats, channel_ids):
        active_f = active_per_frame.float()
        count = active_f.sum(dim=-1, keepdim=True)
        weight = active_f / count.clamp(min=1.0)
        frame_tokens = torch.einsum('btn,bnd->btd', weight, inst_tokens)

        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        time_emb = self.time_proj(time_feats)
        count_norm = torch.log1p(count) / math.log1p(20.0)
        count_emb = self.count_proj(count_norm)
        return frame_tokens + ch_emb + time_emb + count_emb

    def predict_heatmap(self, frame_tokens):
        proj = self.frame_proj(frame_tokens)
        heatmap = torch.einsum('btd,dhw->bthw', proj, self.spatial_codebook)
        heatmap = heatmap * self.logit_scale + self.heatmap_bias
        return heatmap

    def forward(self, channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt,
                inst_scalars, active_per_frame, time_feats, frame_mask):
        inst_tokens, _ = self.encode_inst(
            channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars)
        frame_tokens = self.aggregate_frames(
            inst_tokens, active_per_frame, time_feats, channel_ids)
        heatmap_logits = self.predict_heatmap(frame_tokens)
        return heatmap_logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

if "channel2id" in ckpt:
    ckpt_ch2id = ckpt["channel2id"]
    if len(ckpt_ch2id) != len(channel2id):
        channel2id = ckpt_ch2id
        channels = sorted(channel2id.keys())
        eval_samples = [s for s in eval_samples if s["channel"] in channel2id]
        print(f"✅ 학습 채널 필터링 후 eval: {len(eval_samples):,}개")

model = MergedLayoutModel(n_channels=len(channels)).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"✅ 모델 로드: {CKPT_PATH}")
if "epoch" in ckpt:
    print(f"   epoch={ckpt['epoch']}")
if "metrics" in ckpt:
    print(f"   metrics={ckpt['metrics']}")
if "best_th" in ckpt:
    print(f"   best_th={ckpt['best_th']:.2f}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024


로드: 100%|██████████| 66400/66400 [00:12<00:00, 5316.63it/s]


✅ 채널: 67개 (sampled from 664)
✅ eval 영상: 335개
✅ 모델 로드: ./model/8_layout_1_merged/best.pt
   epoch=28
   metrics={'P': 0.5667025638297226, 'R': 0.6191667720982758, 'F1': 0.5786533512153283, 'IoU': 0.4186812974682888}
   best_th=0.43


In [2]:
# %% 셀 2: 헬퍼 함수 (prepare_inputs / run_inference / compute_eval_metrics / viz)
import cv2
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

plt.rc('font', family='NanumGothic')

OUTPUT_DIR = "./data/8_layout_1_test"
VIS_FPS = 10
VIS_W = 288
VIS_H = 512
GAP = 6
HEADER_H = 50
N_PANELS = 5
total_w = VIS_W * N_PANELS + GAP * (N_PANELS - 1)
total_h = VIS_H + HEADER_H
PANEL_NAMES = ["Original", "GT BBox", "GT Merged", "Pred Heatmap", "Pred Binary"]
EVAL_THRESHOLDS = [i / 100 for i in range(5, 96, 5)]

sx = VIS_W / GRID_W
sy = VIS_H / GRID_H

cmap_heat = matplotlib.colormaps["coolwarm"]

try:
    PIL_FONT = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 20)
    PIL_FONT_SM = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 16)
except:
    PIL_FONT = ImageFont.load_default()
    PIL_FONT_SM = ImageFont.load_default()


def get_instance_colors(n_inst):
    colors = []
    for i in range(n_inst):
        hue = (i * 0.618033988749895) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 0.95)
        colors.append((int(r * 255), int(g * 255), int(b * 255)))
    return colors


def get_swap_channel(orig_channel, video_name, candidates):
    h = hashlib.sha256(f"{orig_channel}||{video_name}".encode()).hexdigest()
    seed = int(h[:8], 16)
    r = random.Random(seed)
    pool = [c for c in candidates if c != orig_channel]
    return r.choice(pool)


def prepare_inputs(sample):
    """Train Dataset과 일치하는 입력 생성."""
    instances = sample["instances"]
    duration = max(sample["duration"], 0.1)
    n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
    n_inst = len(instances)
    times = np.arange(n_frames, dtype=np.float32) / FPS

    if n_inst == 0:
        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)
        return {
            "active_per_frame": np.zeros((n_frames, 0), dtype=np.bool_),
            "merged_mask": np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32),
            "active_matrix": np.zeros((n_frames, 0), dtype=np.bool_),
            "inst_diff_ch": torch.zeros(0, EMB_DIM),
            "inst_diff_vid": torch.zeros(0, EMB_DIM),
            "inst_diff_stt": torch.zeros(0, EMB_DIM),
            "inst_scalars": np.zeros((0, SCALAR_DIM), dtype=np.float32),
            "time_feats": time_feats,
            "n_frames": n_frames,
        }

    inst_starts = np.array([inst["start"] for inst in instances])
    inst_ends   = np.array([inst["end"]   for inst in instances])
    inst_tlens  = np.array([inst["text_len"] for inst in instances])
    inst_cx = np.array([inst["x"] for inst in instances])
    inst_cy = np.array([inst["y"] for inst in instances])
    inst_w  = np.array([inst["w"] for inst in instances])
    inst_h  = np.array([inst["h"] for inst in instances])

    inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
    inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
    inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
    inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

    channel_emb = F.normalize(text2emb.get(sample["channel"], ZERO_EMB), dim=-1)
    video_emb   = F.normalize(text2emb.get(sample["video_name"], ZERO_EMB), dim=-1)
    inst_embs   = torch.stack([text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
    inst_embs   = F.normalize(inst_embs, dim=-1)

    inst_diff_ch  = inst_embs - channel_emb.unsqueeze(0)
    inst_diff_vid = inst_embs - video_emb.unsqueeze(0)
    inst_diff_stt = torch.zeros(n_inst, EMB_DIM)

    ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
    vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

    stt_sims = np.zeros(n_inst, dtype=np.float32)
    has_stts = np.zeros(n_inst, dtype=np.float32)
    stt_segments = sample["stt_segments"]
    if len(stt_segments) > 0:
        stt_starts = np.array([seg["start"] for seg in stt_segments])
        stt_ends   = np.array([seg["end"]   for seg in stt_segments])
        stt_embs_raw = torch.stack([text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
        stt_embs = F.normalize(stt_embs_raw, dim=-1)
        for i in range(n_inst):
            mid = (inst_starts[i] + inst_ends[i]) / 2
            stt_active = (stt_starts <= mid) & (stt_ends > mid)
            stt_active_idx = np.where(stt_active)[0]
            if len(stt_active_idx) > 0:
                inst_diff_stt[i] = inst_embs[i] - stt_embs[stt_active_idx[0]]
                stt_sims[i] = F.cosine_similarity(
                    inst_embs[i].unsqueeze(0), stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                has_stts[i] = 1.0

    active_matrix = (inst_starts[None, :] <= times[:, None] + 0.05) & \
                    (inst_ends[None, :] > times[:, None])

    co_active_per_frame = active_matrix.sum(axis=1)
    inst_avg_coactive = np.zeros(n_inst, dtype=np.float32)
    for i in range(n_inst):
        frames_i = active_matrix[:, i]
        if frames_i.any():
            inst_avg_coactive[i] = co_active_per_frame[frames_i].mean()
    inst_avg_coactive = np.log1p(inst_avg_coactive) / np.log1p(20.0)

    inst_scalars = np.zeros((n_inst, SCALAR_DIM), dtype=np.float32)
    inst_scalars[:, 0] = inst_tlens / MAX_TEXT_LEN
    inst_scalars[:, 1] = ch_sims
    inst_scalars[:, 2] = vid_sims
    inst_scalars[:, 3] = stt_sims
    inst_scalars[:, 4] = has_stts
    inst_scalars[:, 5] = inst_starts / max(duration, 0.1)
    inst_scalars[:, 6] = inst_ends / max(duration, 0.1)
    inst_scalars[:, 7] = (inst_ends - inst_starts) / max(duration, 0.1)
    inst_scalars[:, 8] = inst_avg_coactive

    merged_mask = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
    for j in range(n_inst):
        x0, y0, x1, y1 = int(inst_x0[j]), int(inst_y0[j]), int(inst_x1[j]), int(inst_y1[j])
        if x1 <= x0 or y1 <= y0:
            continue
        active_fr = np.where(active_matrix[:, j])[0]
        if len(active_fr) > 0:
            merged_mask[active_fr[:, None, None],
                        np.arange(y0, y1)[None, :, None],
                        np.arange(x0, x1)[None, None, :]] = 1.0

    time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
    t_norm = times / max(duration, 1.0)
    time_feats[:, 0] = t_norm
    time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
    time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

    return {
        "active_per_frame": active_matrix.astype(np.bool_),
        "merged_mask": merged_mask,
        "active_matrix": active_matrix,
        "inst_diff_ch": inst_diff_ch,
        "inst_diff_vid": inst_diff_vid,
        "inst_diff_stt": inst_diff_stt,
        "inst_scalars": inst_scalars,
        "time_feats": time_feats,
        "n_frames": n_frames,
    }


def run_inference(inputs, cond_channel_id):
    """Model 1 — single forward (no chunking needed)."""
    n_frames = inputs["n_frames"]

    if inputs["inst_diff_ch"].shape[0] == 0:
        return {"merged_pred": np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)}

    cid = torch.tensor([cond_channel_id], dtype=torch.long, device=device)
    fm = torch.ones(1, n_frames, dtype=torch.bool, device=device)
    dc = inputs["inst_diff_ch"].unsqueeze(0).to(device)
    dv = inputs["inst_diff_vid"].unsqueeze(0).to(device)
    ds = inputs["inst_diff_stt"].unsqueeze(0).to(device)
    sc = torch.from_numpy(inputs["inst_scalars"]).unsqueeze(0).to(device)
    apf = torch.from_numpy(inputs["active_per_frame"]).unsqueeze(0).to(device)
    tf = torch.from_numpy(inputs["time_feats"]).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16,
                                 enabled=(device.type == "cuda")):
            logits = model(cid, dc, dv, ds, sc, apf, tf, fm)
            prob = torch.sigmoid(logits.float())

    return {"merged_pred": prob[0].cpu().numpy()}      # (T, H, W)


def find_best_threshold(merged_pred, merged_gt):
    """IoU 기준 best threshold (0.05~0.95)."""
    gt = (merged_gt > 0.5)
    n_gt = gt.sum()
    best_th = 0.5
    best_iou = -1.0
    for th in EVAL_THRESHOLDS:
        pred = merged_pred > th
        tp = (pred & gt).sum()
        fp = (pred & ~gt).sum()
        fn = n_gt - tp
        iou = tp / (tp + fp + fn + 1e-8)
        if iou > best_iou:
            best_iou = iou
            best_th = th
    return best_th


def compute_eval_metrics(merged_pred, merged_gt, threshold, active_matrix):
    """Pixel-level metrics on merged heatmap."""
    pred_bin = merged_pred > threshold
    gt_bin = merged_gt > 0.5

    tp_all = int((pred_bin & gt_bin).sum())
    fp_all = int((pred_bin & ~gt_bin).sum())
    fn_all = int((~pred_bin & gt_bin).sum())
    p_all = tp_all / (tp_all + fp_all + 1e-8)
    r_all = tp_all / (tp_all + fn_all + 1e-8)
    f1_all = 2 * p_all * r_all / (p_all + r_all + 1e-8)
    iou_all = tp_all / (tp_all + fp_all + fn_all + 1e-8)

    gt_pos = int(gt_bin.sum())
    pred_pos = int(pred_bin.sum())
    coverage = pred_pos / (gt_pos + 1e-8)

    n_frames = merged_pred.shape[0]
    frame_metrics = []
    for fi in range(n_frames):
        pf = pred_bin[fi]
        gf = gt_bin[fi]
        tp = int((pf & gf).sum())
        fp = int((pf & ~gf).sum())
        fn = int((~pf & gf).sum())
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
        n_active = int(active_matrix[fi].sum()) if active_matrix.shape[1] > 0 else 0
        frame_metrics.append({
            "frame": fi, "t_sec": round(fi / FPS, 1), "n_active": n_active,
            "gt_positive": int(gf.sum()), "pred_positive": int(pf.sum()),
            "P": round(float(p), 4), "R": round(float(r), 4),
            "F1": round(float(f1), 4), "IoU": round(float(iou), 4),
        })

    return {
        "aggregate": {
            "P": round(float(p_all), 4), "R": round(float(r_all), 4),
            "F1": round(float(f1_all), 4), "IoU": round(float(iou_all), 4),
            "coverage_ratio": round(float(coverage), 4),
            "gt_positive_total": gt_pos, "pred_positive_total": pred_pos,
            "threshold": float(threshold),
            "method": "merged_layout_best_th",
        },
        "per_frame": frame_metrics,
    }


def find_frame_dir(sample):
    return os.path.join(FRAME_DIR, sample["channel"], sample["file_id"])


def draw_bboxes_on_panel(pil_img, x_offset, bboxes, colors):
    draw = ImageDraw.Draw(pil_img)
    for (bx0, by0, bx1, by1, inst_idx) in bboxes:
        color = colors[inst_idx % len(colors)]
        px0 = x_offset + bx0 * sx
        py0 = HEADER_H + by0 * sy
        px1 = x_offset + bx1 * sx
        py1 = HEADER_H + by1 * sy
        draw.rectangle([px0, py0, px1, py1], outline=color, width=2)


def save_eval_data(json_path, sample, cond_channel, metrics, threshold):
    record = {
        "orig_channel": sample["channel"],
        "video_name": sample["video_name"],
        "file_id": sample["file_id"],
        "cond_channel": cond_channel,
        "n_instances": len(sample["instances"]),
        "duration": sample["duration"],
        "n_frames": len(metrics["per_frame"]),
        "model_ckpt": CKPT_PATH,
        "threshold": float(threshold),
        "aggregate": metrics["aggregate"],
        "per_frame": metrics["per_frame"],
    }
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)


print("✅ 헬퍼 함수 정의 완료")

✅ 헬퍼 함수 정의 완료


In [6]:
# %% 셀 3: 랜덤 1개 영상 — 5패널 시각화
candidates = [s for s in eval_samples if len(s["instances"]) > 0]
sample = random.choice(candidates)
orig_ch = sample["channel"]
file_id = sample["file_id"]
instances = sample["instances"]
n_inst = len(instances)

print(f"🎲 선택: [{orig_ch}] {file_id}")
print(f"   인스턴스: {n_inst}개  duration: {sample['duration']:.1f}s")

inputs = prepare_inputs(sample)
n_frames = inputs["n_frames"]
active_matrix = inputs["active_matrix"]
merged_gt = inputs["merged_mask"]

inst_colors = get_instance_colors(n_inst)
orig_ch_id = channel2id[orig_ch]

result = run_inference(inputs, orig_ch_id)
merged_pred = result["merged_pred"]

best_th = find_best_threshold(merged_pred, merged_gt)
metrics = compute_eval_metrics(merged_pred, merged_gt, best_th, active_matrix)
agg = metrics["aggregate"]
print(f"   th={best_th:.2f}  P={agg['P']:.3f}  R={agg['R']:.3f}  F1={agg['F1']:.3f}  IoU={agg['IoU']:.3f}")

inst_bboxes = []
for j in range(n_inst):
    cx, cy = int(instances[j]["x"]), int(instances[j]["y"])
    w, h = int(instances[j]["w"]), int(instances[j]["h"])
    x0 = max(0, cx - w // 2)
    y0 = max(0, cy - h // 2)
    x1 = min(GRID_W, x0 + w)
    y1 = min(GRID_H, y0 + h)
    inst_bboxes.append((x0, y0, x1, y1))

frame_dir = find_frame_dir(sample)
out_path = "./data/8_layout_1_test/_random_5panel.mp4"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(out_path, fourcc, VIS_FPS, (total_w, total_h))

px = [i * (VIS_W + GAP) for i in range(N_PANELS)]

for fi in range(n_frames):
    canvas = np.zeros((total_h, total_w, 3), dtype=np.uint8)

    # Panel 1: Original
    frame_path = os.path.join(frame_dir, f"frame_{fi+1:08d}.jpg")
    if os.path.exists(frame_path):
        orig = cv2.resize(cv2.imread(frame_path), (VIS_W, VIS_H))
    else:
        orig = np.zeros((VIS_H, VIS_W, 3), dtype=np.uint8)
    canvas[HEADER_H:, px[0]:px[0]+VIS_W] = orig

    # Panel 3: GT Merged (white mask)
    gt_resized = cv2.resize(merged_gt[fi], (VIS_W, VIS_H), interpolation=cv2.INTER_NEAREST)
    gt_panel = np.stack([gt_resized * 255]*3, axis=-1).astype(np.uint8)
    canvas[HEADER_H:, px[2]:px[2]+VIS_W] = gt_panel

    # Panel 4: Pred Heatmap (coolwarm)
    pred_resized = cv2.resize(merged_pred[fi], (VIS_W, VIS_H), interpolation=cv2.INTER_LINEAR)
    pred_panel = (cmap_heat(pred_resized)[:, :, :3] * 255).astype(np.uint8)
    canvas[HEADER_H:, px[3]:px[3]+VIS_W] = cv2.cvtColor(pred_panel, cv2.COLOR_RGB2BGR)

    # Panel 5: Pred Binary (after threshold)
    pred_bin = (merged_pred[fi] > best_th).astype(np.float32)
    pred_bin_resized = cv2.resize(pred_bin, (VIS_W, VIS_H), interpolation=cv2.INTER_NEAREST)
    pred_bin_panel = np.stack([pred_bin_resized * 255]*3, axis=-1).astype(np.uint8)
    canvas[HEADER_H:, px[4]:px[4]+VIS_W] = pred_bin_panel

    pil_img = Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)

    for i, name in enumerate(PANEL_NAMES):
        draw.text((px[i] + 6, 5), name, font=PIL_FONT, fill=(255, 255, 255))

    t_sec = fi / FPS
    n_active = int(active_matrix[fi].sum()) if active_matrix.shape[1] > 0 else 0
    draw.text((6, 28),
              f"t={t_sec:.1f}s  active={n_active}  th={best_th:.2f}  [{orig_ch}]  F1={agg['F1']:.3f}",
              font=PIL_FONT_SM, fill=(180, 180, 180))

    # Panel 2: GT BBox
    if active_matrix.shape[1] > 0:
        active_inst = np.where(active_matrix[fi])[0]
        gt_boxes = [(inst_bboxes[j][0], inst_bboxes[j][1],
                      inst_bboxes[j][2], inst_bboxes[j][3], j) for j in active_inst]
        draw_bboxes_on_panel(pil_img, px[1], gt_boxes, inst_colors)

    canvas = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    writer.write(canvas)

writer.release()
print(f"✅ 저장: {out_path} ({n_frames} frames)")

🎲 선택: [딩고 스토리 _ dingo story] 햄버거 먹으러 갔는데 최애가 알바생이라면？!__l0DqzljhvIU
   인스턴스: 40개  duration: 59.5s
   th=0.30  P=0.949  R=0.742  F1=0.833  IoU=0.713
✅ 저장: ./data/8_layout_1_test/_random_5panel.mp4 (596 frames)
